# CSCI450 Project 1
## Pill Counting
[description]

By Sergei and Lucy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, feature, color, exposure, filters, morphology, measure
# from skimage.transform import resize
import os

from skimage.filters import gaussian, median

### Segmentation

In [ ]:
def segment_pills(image, sigma=1, min_size=150,hole_area=50):
    gray = color.rgb2gray(image)
    smooth = gaussian(gray, sigma=sigma)
    smooth = median(smooth)

    threshold = filters.threshold_otsu(smooth)

    mask = smooth < threshold

    mask = morphology.remove_small_objects(mask, min_size=min_size)
    mask = morphology.remove_small_holes(mask, area_threshold=hole_area)
    #each pill gets different number
    labels = measure.label(mask)

    img_segmented = image.copy()
    img_segmented[~mask] = 0

    return img_segmented, mask, labels


In [ ]:
#testing if working
img_00000 = io.imread("img_00000.png")
image_segmented, mask, labels = segment_pills(img_00000)

In [ ]:
plt.figure(figsize=(15,5))

fig, (ax1, ax2, ax3) = plt.subplots(1,3,figsize = (10,5))
ax1.imshow(img_00000)
ax1.set_title('Original Image')
ax1.axis('off')

ax2.imshow(mask, cmap='gray')
ax2.set_title('Segmentation Masks')
ax2.axis('off')

ax3.imshow(image_segmented)
ax3.set_title('Segmented Pills')
ax3.axis('off')

plt.tight_layout()
plt.show()

### Feature extraction

In [2]:
path = 'pill_dataset/images'
imagelist = os.listdir(path)

feature_color_hist = []
feature_hog = []

In [3]:
for i in range(len(imagelist)):
    img = io.imread(path + '/' + imagelist[i])
    #resized_img = resize(img, (HEIGHT, WIDTH), anti_aliasing=True)
    norm_image = exposure.rescale_intensity(img, in_range='image', out_range=(0,255)) # our norm func. works per channel
    hsv_img = color.rgb2hsv(norm_image)

    hog_fd = feature.hog(norm_image, orientations=9, pixels_per_cell=(16, 16),
                cells_per_block=(1, 1), channel_axis=2)
    hist, bins = exposure.histogram(hsv_img[:,:,0], nbins=256)
    
    line = '\rProcessing '+ str(i+1) + ' of '+ str(len(imagelist))
    print(line, end='')

Processing 5 of 15000

KeyboardInterrupt: 